# EduHinglish: Gemma-3-4B-it Fine-Tuning (T4 GPU)
This notebook fine-tunes Google's Gemma-3-4B-it model on the combined Biology and Social Science Hinglish dataset using QLoRA.

> **Runtime**: Runtime → Change runtime type → **T4 GPU**

## 0. Install Dependencies
Pin versions to avoid API-breaking updates mid-run. **Restart the runtime after this cell completes.**

In [ ]:
!pip install -q -U \
    "bitsandbytes>=0.43.0" \
    "transformers>=4.47.0" \
    "peft>=0.14.0" \
    "accelerate>=1.0.0" \
    "datasets>=3.0.0" \
    "trl>=0.12.0,<0.20.0" \
    "huggingface_hub"
print("Done. Please RESTART the runtime (Runtime -> Restart session), then run the remaining cells.")

## 1. Authentication & Drive Mount
Gemma is a gated model — you must accept its license first.
1. Go to https://huggingface.co/google/gemma-3-4b-it and click **Acknowledge license**.
2. Go to https://huggingface.co/settings/tokens and create a **Read** token.
3. In Colab, open **Secrets** (key icon in left sidebar) → add `HF_TOKEN` → paste your token → enable **Notebook access**.

In [ ]:
from google.colab import drive, userdata
from huggingface_hub import login

drive.mount('/content/drive')

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print("HuggingFace login successful.")

## 2. Load and Prepare Dataset

In [ ]:
import json
import random
from datasets import Dataset

base_dir = "/content/drive/MyDrive/EduHinglish-ollama"

try:
    with open(f"{base_dir}/data/bio_9_10.json", "r") as f:
        bio_data = json.load(f)
    with open(f"{base_dir}/data/ss9.json", "r") as f:
        ss9_data = json.load(f)
    print(f"bio_9_10.json: {len(bio_data)} entries")
    print(f"ss9.json     : {len(ss9_data)} entries")
except FileNotFoundError as e:
    print(f"ERROR: {e}")
    print(f"Please ensure your files are at {base_dir}/data/")
    bio_data, ss9_data = [], []

all_data = bio_data + ss9_data
random.seed(42)
random.shuffle(all_data)
print(f"\nTotal entries loaded: {len(all_data)}")

### Format for Gemma Chat Template

In [ ]:
def format_gemma_chat(entry):
    context  = entry.get("ncert_context", "")
    question = entry.get("question_english", "")
    intent   = entry.get("intent", "")

    prompt = (
        f"Context:\n{context}\n\n"
        f"Question:\n{question}\n\n"
        f"Intent Type: {intent}\n"
        f"Please provide the explanation in Hinglish."
    )
    answer = entry.get("answer_hinglish") or entry.get("answer_english", "")

    formatted_text = (
        f"<start_of_turn>user\n{prompt}<end_of_turn>\n"
        f"<start_of_turn>model\n{answer}<end_of_turn>"
    )
    return {"text": formatted_text}

formatted_data = [format_gemma_chat(e) for e in all_data]
dataset = Dataset.from_list(formatted_data)
print(f"Formatted {len(dataset)} entries.")

### Train / Test Split (95 / 5)

In [ ]:
split_dataset = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = split_dataset["train"]
eval_dataset  = split_dataset["test"]

print(f"Training samples  : {len(train_dataset)}")
print(f"Evaluation samples: {len(eval_dataset)}")
print("\nSample entry:")
print(train_dataset[0]["text"])

## 3. Load Tokenizer & Model (QLoRA for T4)
4-bit NF4 quantisation with `fp16` — T4 does **not** support bfloat16.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "google/gemma-3-4b-it"

# --- Tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # required for causal LM training

# --- 4-bit quantisation config ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# --- Load base model ---
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False       # must disable when using gradient checkpointing
model.enable_input_require_grads()   # allows LoRA layers to receive gradients

print(f"Model loaded. dtype={model.dtype}")

### LoRA Adapters

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 4. Training
Checkpoints are saved every 100 steps to Drive.

> **API note (trl >= 0.9):** SFT-specific args (`dataset_text_field`, `max_length`, `packing`) live in `SFTConfig`, not in `SFTTrainer`. `max_seq_length` was renamed to `max_length`. Use `processing_class` instead of the deprecated `tokenizer` kwarg.

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    # Output dir on Drive
    output_dir=f"{base_dir}/training/gemma-eduhinglish-v1",

    # SFT-specific args (must be here, NOT passed directly to SFTTrainer)
    dataset_text_field="text",
    max_length=1024,            # renamed from max_seq_length in trl >= 0.20
    packing=False,

    # Batch / accumulation
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,

    # Gradient checkpointing
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    # Optimisation
    learning_rate=2e-4,
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    group_by_length=True,

    # Precision — T4 uses fp16, NOT bf16
    fp16=True,
    bf16=False,

    # Steps
    max_steps=500,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,

    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,   # replaces the deprecated tokenizer= kwarg
    args=sft_config,
)

print("Starting Training...")
trainer.train()

## 5. Save Adapter & Test Inference

In [ ]:
adapter_path = f"{base_dir}/training/gemma-eduhinglish-adapter"
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Adapter saved to: {adapter_path}")

In [ ]:
model.eval()

def generate_response(prompt_text, max_new_tokens=200):
    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    input_len = inputs["input_ids"].shape[1]
    # Decode only newly generated tokens (skip echoed prompt)
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

test_entry  = eval_dataset[0]["text"]
prompt_only = test_entry.split("<start_of_turn>model")[0] + "<start_of_turn>model\n"

print("--- PROMPT ---")
print(prompt_only)
print("\n--- GENERATED HINGLISH RESPONSE ---")
print(generate_response(prompt_only))